# 1. Import dataset

## 1.1 Import safe/ vulnerable dataset and attach labels, in particular
+ Safe package -> 0
+ vulnerable package -> 1

In [138]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # or project root

In [139]:
import pandas as pd
import random

In [140]:
df_safe = pd.read_csv("../data/safe_packages.txt")
df_safe["isRisky"] = 0
print(df_safe.head(5))
print(f"Imported safe packages into dataframe with shape {df_safe.shape}")


df_vulnerable = pd.read_csv("../data/vulnerable_packages.txt")
df_vulnerable["isRisky"] = 1
print(df_vulnerable.head(5))
print(f"Imported vulnerable packages into dataframe with shape {df_vulnerable.shape}")

df = pd.concat([df_safe, df_vulnerable], ignore_index=True)
print(df.head(5))
print(f"Shape of combined dataframe: {df.shape}")

X = df.drop(['isRisky'], axis=1)
y = df['isRisky']

                                package_name@version  isRisky
0                                       komlib@0.1.0        0
1                          juegodeguerrajavier@7.1.2        0
2  odoo12-addon-mail-outbound-static@12.0.1.0.1.9...        0
3                                   momba@1.0.0.dev4        0
4                                    homestock@1.6.8        0
Imported safe packages into dataframe with shape (4786, 2)
               aim@2.5.0  isRisky
0              aim@2.7.3        1
1              aim@3.0.2        1
2          aiohttp@3.6.0        1
3        aiohttp@3.6.0a4        1
4  aiohttp-session@0.3.0        1
Imported vulnerable packages into dataframe with shape (2218, 2)
                                package_name@version  isRisky aim@2.5.0
0                                       komlib@0.1.0        0       NaN
1                          juegodeguerrajavier@7.1.2        0       NaN
2  odoo12-addon-mail-outbound-static@12.0.1.0.1.9...        0       NaN
3           

## 1.2 Split training and testing set

In [121]:
from sklearn.model_selection import train_test_split

In [122]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y,random_state=42)
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (7697, 2)
Shape of X_test: (1925, 2)
Shape of y_train: (7697,)
Shape of y_test: (1925,)


## 2 Training Loop - Test: using one iteration

In [135]:
from scripts.GraphGenerator import GraphGenerator
from scripts.FeatureGenerator import FeatureGenerator

In [136]:
feature_generator_pypi = FeatureGenerator('pypi')

In [137]:
# Get a package@version from X_train 
pkg = X_train['package_name@version'].values[0]
package_name, version = pkg.rsplit('@', 1)
graph_generator = GraphGenerator(package_name, version)


[GraphGenerator] Initializing 1st layer of the graph...
[GraphGenerator] Fetching dependencies...
Level: 3
[['fluffy-disco@1.0.0']]


In [126]:
import networkx as nx
from pyvis.network import Network

In [127]:
G = nx.DiGraph()

for package_id, node in graph_generator.nodes_map.items():
    G.add_node(package_id)
    for dep_id in node.depends_on:
        G.add_edge(package_id, dep_id)

net = Network(height="900px", width="100%", directed=True, notebook=False)
net.from_nx(G)
net.write_html("dependency_graph.html", notebook=False)

In [128]:
import importlib, inspect
import scripts.FeatureGenerator as fg_mod
print(fg_mod.__file__)
importlib.reload(fg_mod)
from scripts.FeatureGenerator import FeatureGenerator
feature_generator_pypi = FeatureGenerator("pypi")


/Users/xipingye/software-supply-chain-risk-detector/scripts/FeatureGenerator.py


In [129]:
for p in graph_generator.nodes_map:
    res = feature_generator_pypi.get_full_features(p, graph_generator.nodes_map)
    graph_generator.nodes_map[p].features = res['full_metadata']
    print(res['full_metadata'])
    print(res['col_names'])

[1, 0, False, False, True, True, True, False, 1, 1, 0, 0]
['num_authors', 'num_maintainers', 'has_license', 'yanked', 'has_project_url', 'has_package_url', 'has_release_url', 'has_organization', 'num_roles', 'num_distributions', 'in_degree', 'out_degree']


In [130]:
import torch

In [131]:
node_ids = list(graph_generator.nodes_map.keys())
node_to_idx = {nid: i for i, nid in enumerate(node_ids)}

x_tensor = torch.tensor(
    [graph_generator.nodes_map[nid].features for nid in node_ids],
    dtype=torch.float
)

print(f"Shape of tensor: {x_tensor.shape}")

Shape of tensor: torch.Size([1, 12])


In [132]:
edges = []

for nid, node in graph_generator.nodes_map.items():
    src = node_to_idx[nid]
    for dep in node.depends_on:
        if dep in node_to_idx:  # ensure inside subgraph
            dst = node_to_idx[dep]
            edges.append([src, dst])

edge_index = torch.tensor(edges, dtype=torch.long).t()

In [133]:
print(edges)

[]


### What if there is no edges? Independent Package

In [134]:
edge_index = torch.cat([edge_index, edge_index[[1, 0]]], dim=1)

IndexError: index is out of bounds for dimension with size 0

In [ ]:
print(edge_index)

tensor([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   1,   3,
           6,   6,   6,   6,   6,   6,   6,   6,   8,  11,  12,  13,  13,  14,
          14,  14,  14,  16,  17,  17,  19,  19,  19,  21,  21,  22,  22,  22,
          22,  22,  22,  23,  23,  23,  23,  23,  23,  23,  24,  25,  25,  26,
          26,  27,  27,  28,  28,  29,  29,  30,  30,  31,  31,  31,  31,  32,
          32,  33,  33,  33,  33,  34,  34,  35,  36,  36,  36,  37,  37,  37,
          37,  37,  38,  38,  39,  39,  40,  40,  40,  41,  41,  42,  42,  43,
          43,  43,  43,  43,  43,  43,  43,  43,  43,  43,  43,  45,  46,  47,
          47,  47,  47,  48,  48,  48,  48,  48,  48,  48,  48,  50,  50,  50,
          51,  51,  51,  53,  54,  57,  57,  58,  58,  58,  58,  59,  59,  59,
          59,  60,  60,  69,  73,  73,  76,  77,  82

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

In [ ]:
class GNN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = nn.Linear(hidden_dim, out_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        return x  # node embeddings

In [ ]:
label = y_train.values[0]
target_node_idx = 0 

In [ ]:
model = GNN(in_dim=x_tensor.shape[1], hidden_dim=64, out_dim=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()

# Forward
node_embeddings = model(x_tensor, edge_index)

# Only use ROOT node
target_embedding = node_embeddings[target_node_idx]

# Classification
logits = model.lin(target_embedding)

# Loss
label_tensor = torch.tensor([label], dtype=torch.long)
loss = F.cross_entropy(logits.unsqueeze(0), label_tensor)

# Backward
optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Loss:", loss.item())

Loss: 1.1323375701904297
